In [1]:
# Setup
import torch
import copy
import json

from utils.data_reader import load_and_prepare_time_series_data
from utils.evaluator import Evaluator
from models.baseline_models import LSTMModel, BiLSTMModel, GRUModel
from models.custom_models import MGSSMModel, MGSSMsModel, ExtendedMGSSMsModel

In [2]:

selected_country_codes = ['US', 'IN', 'BR', 'FR', 'DE',
                         'GB', 'RU', 'IT', 'TR', 'ES',
                         'VN', 'AR', 'AU', 'AT', 'BD',
                         'BE', 'BG', 'CA', 'CL', 'CN',
                         'CU', 'DK', 'FI', 'GE', 'GR',
                         'ID', 'JP', 'JO', 'KE', 'KR',
                         'LR', 'MY','ML', 'MX', 'NL',
                         'NO', 'PH','SE', 'CH', 'TH']

# Fetch and prepare the data for each country code
with open("/home/theppawan/nn-models/data/COVID19_url_data.json", "r") as f:
    dataset = json.load(f)

train_loader_dist = {}
val_loader_dist = {}
test_loader_dist = {}
scaler_dist = {}
for key in selected_country_codes:
    train_loader, val_loader, test_loader, scaler = load_and_prepare_time_series_data(
        filepath_or_url = dataset['region'].format(region=key),
        target_column=['cumulative_confirmed'],
        date_column="date",
        seq_length=14,
        batch_size=64,
        train_split=0.8,
        fill_missing=True)
    train_loader_dist[key] = train_loader
    val_loader_dist[key] = val_loader
    test_loader_dist[key] = test_loader
    scaler_dist[key] = scaler

# Setup model configurations
model_config_dict = {
    "Baseline LSTM": LSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline BiLSTM": BiLSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline GRU": GRUModel(input_size=1, hidden_size=128, num_layers=1, output_size=1),
    "MGSSM": MGSSMModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "MGSSMs": MGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "ExtendedMGSSMs": ExtendedMGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32, p=2)
}

Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/US.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/IN.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/BR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/FR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data

In [3]:
trained_model_dict = {key: {} for key in selected_country_codes}
# loading the checkpointed models
for model_name, model_config in model_config_dict.items():
    # analyze the model from the checkpoint
    for key in selected_country_codes:
        country_specific_model = copy.deepcopy(model_config)
        country_specific_model.load_state_dict(torch.load(f"checkpoints/best_{model_config.__class__.__name__}_{key}.pth"))
        trained_model_dict[key][model_name] = country_specific_model

evaluator = Evaluator()
for key in selected_country_codes:
    val_loader = val_loader_dist[key]
    scaler = scaler_dist[key]
    evaluator.compare_models(trained_model_dict[key], val_loader, scaler)

Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating mode

In [4]:
performance_data = evaluator.build_performance_dataframe(nested_models_dict=trained_model_dict, data_loaders_dict=val_loader_dist, scalers_dict=scaler_dist, target_metric="MSE")
performance_data

Evaluating models for dataset: US...
Evaluating models for dataset: IN...
Evaluating models for dataset: BR...
Evaluating models for dataset: FR...
Evaluating models for dataset: DE...
Evaluating models for dataset: GB...
Evaluating models for dataset: RU...
Evaluating models for dataset: IT...
Evaluating models for dataset: TR...
Evaluating models for dataset: ES...
Evaluating models for dataset: VN...
Evaluating models for dataset: AR...
Evaluating models for dataset: AU...
Evaluating models for dataset: AT...
Evaluating models for dataset: BD...
Evaluating models for dataset: BE...
Evaluating models for dataset: BG...
Evaluating models for dataset: CA...
Evaluating models for dataset: CL...
Evaluating models for dataset: CN...
Evaluating models for dataset: CU...
Evaluating models for dataset: DK...
Evaluating models for dataset: FI...
Evaluating models for dataset: GE...
Evaluating models for dataset: GR...
Evaluating models for dataset: ID...
Evaluating models for dataset: JP...
E

,Baseline LSTM,Baseline BiLSTM,Baseline GRU,MGSSM,MGSSMs,ExtendedMGSSMs
US,3.275813e+10,2.931701e+10,4.189532e+09,1.532140e+11,3.118342e+10,3.956473e+10
IN,1.001611e+08,1.809958e+07,5.436851e+07,7.340424e+08,1.349523e+08,5.507878e+07
BR,6.419532e+08,3.175263e+08,1.102976e+09,6.619543e+09,1.096989e+10,1.960648e+09
FR,8.544586e+12,5.074134e+12,9.218955e+12,5.379041e+12,4.287771e+12,4.080967e+12
DE,7.664578e+09,2.921927e+09,5.748462e+09,7.553129e+09,2.471913e+10,3.706762e+09
GB,1.986474e+09,1.151065e+08,2.411332e+09,2.285068e+09,3.566276e+08,1.597134e+08
RU,3.332873e+08,1.195091e+08,1.044799e+09,3.249092e+08,4.815562e+08,1.544701e+08
IT,8.869321e+09,2.880845e+08,3.368396e+10,1.150071e+09,4.206722e+09,2.049500e+09
TR,3.791880e+09,4.576516e+09,5.167199e+09,3.664900e+09,3.254723e+09,2.992440e+09
ES,2.155764e+08,1.734645e+08,5.772648e+08,5.059242e+08,4.794276e+09,2.749089e+08


In [5]:
friedman_statistic, p_value = evaluator.friedman_test(performance_data)
p_value

np.float64(2.38304347856472e-10)

In [6]:
evaluator.holm_bonferroni_posthoc(performance_data, control_model="ExtendedMGSSMs", alpha=0.05, metric_is_loss=True)

,unadjusted p-value,Step (i),Holm threshold,Holm adjusted p-value,reject null hypothesis
baseline model,,,,,
MGSSMs,8.171992e-08,1,0.010000,4.085996e-07,True
MGSSM,4.890999e-03,2,0.012500,1.956400e-02,True
Baseline GRU,1.876903e-01,3,0.016667,5.630708e-01,False
Baseline LSTM,4.183775e-01,4,0.025000,8.367550e-01,False
Baseline BiLSTM,9.934871e-01,5,0.050000,9.934871e-01,False
